# Lesson 07 - Pola Desain Perencanaan

Notebook ini menunjukkan **Pola Desain Perencanaan** untuk agen AI menggunakan Microsoft Agent Framework.
Anda akan belajar bagaimana memecah permintaan perjalanan yang kompleks menjadi subtugas yang terstruktur, menetapkannya kepada agen spesialis,
dan menjalankan rencana yang dihasilkan — semuanya menggunakan output terstruktur yang didukung oleh model Pydantic.


## Pengaturan


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Dekonstruksi Tugas

Dekonstruksi tugas adalah inti dari pola desain perencanaan. Alih-alih meminta satu agen tunggal untuk menangani permintaan kompleks secara menyeluruh, kita memecah masalah menjadi **subtugas** yang lebih kecil dan terdefinisi dengan baik. Setiap subtugas dialokasikan ke agen spesialis (misalnya, penerbangan, hotel, aktivitas) dengan prioritas dan urutan ketergantungan yang jelas.

Pendekatan ini memberikan beberapa manfaat:
- **Kejelasan**: setiap subtugas memiliki tanggung jawab tunggal.
- **Paralelisme**: subtugas independen dapat berjalan secara bersamaan.
- **Keandalan**: kegagalan terisolasi pada subtugas individual.
- **Pelacakan anggaran**: biaya diperkirakan per subtugas dan dijumlahkan.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Membuat Agen Perencana dengan Output Terstruktur

Agen perencana berperan sebagai **koordinator meja depan**. Diberikan permintaan perjalanan tingkat tinggi, agen ini menghasilkan `TravelPlan` terstruktur — memecah permintaan menjadi subtugas, menetapkan prioritas, dan mengidentifikasi ketergantungan sehingga concierge atau lapisan pelaksana dapat menjalankan pekerjaan tersebut.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Menjalankan Rencana dengan Alat Spesialis

Setelah petugas meja depan membuat rencana yang terstruktur, **agen concierge** akan melaksanakannya.  
Setiap alat spesialis menangani satu kategori sub-tugas (penerbangan, hotel, aktivitas). Concierge  
akan mengulangi sub-tugas rencana sesuai urutan ketergantungan dan mengirimkan masing-masing ke  
alat yang sesuai.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Ringkasan

Dalam pelajaran ini Anda mempelajari **Polanya Desain Perencanaan** untuk agen AI:

1. **Dekompisi Tugas** — Agen perencanaan meja depan memecah permintaan perjalanan yang kompleks menjadi
   subtugas terstruktur menggunakan model Pydantic, menetapkan masing-masing kepada agen spesialis dengan prioritas
   dan ketergantungan.
2. **Output Terstruktur** — Dengan memberikan `response_format` agen mengembalikan objek `TravelPlan` yang tervalidasi
   alih-alih teks bebas, sehingga pemrosesan selanjutnya menjadi andal.
3. **Eksekusi Rencana** — Agen konsier mengiterasi subtugas menggunakan alat spesialis
   (`book_flight`, `reserve_hotel`, `book_activity`) untuk melaksanakan rencana dan melaporkan hasil.

Polanya ini memisahkan *apa yang harus dilakukan* (perencanaan) dari *bagaimana melakukannya* (eksekusi), membuat agen
lebih modular, dapat diuji, dan lebih mudah untuk dikembangkan.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berusaha mencapai ketepatan, harap diingat bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang salah yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
